# 🤖 AI Agent — Tool Testing Notebook

This notebook tests each tool individually before wiring them into the agent.

Always test tools in isolation first — if the agent fails, you know whether
the problem is in the tool or in the agent's reasoning.


## 1. Test Web Search Tool

In [ ]:
from langchain_community.tools import DuckDuckGoSearchRun

search = DuckDuckGoSearchRun()

# Test a simple search
result = search.run("AI jobs Pakistan 2026")
print(result[:500])

## 2. Test Calculator Tool

In [ ]:
import sys
sys.path.append('..')
from src.tools import safe_calculate

# Test various calculations
tests = [
    "15 * 0.15",
    "sqrt(144)",
    "2 ** 10",
    "100000 * (1 + 0.12) ** 3",
    "10 / 0",  # division by zero
]

for expr in tests:
    result = safe_calculate(expr)
    print(f"Expression: {expr}")
    print(f"Result    : {result}")
    print()

## 3. Test Wikipedia Tool

In [ ]:
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

wikipedia = WikipediaQueryRun(
    api_wrapper=WikipediaAPIWrapper(top_k_results=1, doc_content_chars_max=500)
)

result = wikipedia.run("machine learning")
print(result)

## 4. Test Document Reader Tool

In [ ]:
from src.tools import load_document, read_document

# Simulate loading a document
sample_doc = """
Company Overview
Our company was founded in 2020 in Islamabad, Pakistan.
We specialize in fintech solutions and AI-powered products.

Revenue Report 2025
Total revenue for 2025 was PKR 50 million.
This represents a 35% increase from 2024.

Team
We have 45 employees across engineering, design, and operations.
The AI team consists of 8 engineers.
"""

load_document(sample_doc, "company_report.txt")

# Test queries
queries = [
    "When was the company founded?",
    "What was the revenue?",
    "How many AI engineers are there?"
]

for query in queries:
    result = read_document(query)
    print(f"Query : {query}")
    print(f"Result: {result[:200]}")
    print()

## 5. Test the Full Agent

In [ ]:
# Make sure Ollama is running before this cell
from src.agent import build_agent, run_agent
from src.memory import create_memory

memory = create_memory()
agent_executor, memory = build_agent(memory)

# Test 1: Simple question (no tool needed)
result = run_agent(agent_executor, "What is your name?")
print("Q: What is your name?")
print(f"A: {result['output']}")
print(f"Tools used: {len(result['intermediate_steps'])} steps")
print()

In [ ]:
# Test 2: Calculator question
result = run_agent(agent_executor, "What is 15% of 85000?")
print("Q: What is 15% of 85000?")
print(f"A: {result['output']}")
print(f"Tools used: {[step[0].tool for step in result['intermediate_steps']]}")

In [ ]:
# Test 3: Memory test
result1 = run_agent(agent_executor, "My name is Hasan.")
result2 = run_agent(agent_executor, "What is my name?")
print(f"A: {result2['output']}")
print("Memory is working if the agent remembers 'Hasan'")

## 6. Summary

| Tool | Works? | Notes |
|------|--------|-------|
| web_search | ✅ | DuckDuckGo, no API key |
| calculator | ✅ | Safe eval, handles errors |
| wikipedia | ✅ | Top 2 results, 1000 chars |
| document_reader | ✅ | Keyword matching |
| agent (ReAct) | ✅ | LLaMA3 via Ollama |

**Next step:** Run the Streamlit app with `streamlit run app/app.py`
